# Week 6, Day 2 — Building Your Own MCP Server
### Local Models Edition — by Abhishek

Day 1 you *used* MCP servers someone else wrote. Today you write one: an
accounts system for a trading floor, exposed as MCP tools. It's pure local
Python and SQLite — no paid API involved anywhere in this lab.

The implementation lives in `backend/accounts.py` (a plain Python class) and
`backend/accounts_server.py` (the same class wrapped as an MCP server with
`FastMCP`) — have a read of both before running the cells below.


## 0. Setup

In [ ]:
%pip install -q openai-agents mcp


In [ ]:
from openai import AsyncOpenAI
from agents import Agent, Runner, set_default_openai_client, set_tracing_disabled
from agents.mcp import MCPServerStdio

local_client = AsyncOpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
set_default_openai_client(local_client)
set_tracing_disabled(True)
MODEL_NAME = "llama3.2:3b"

import sys
if sys.platform == "win32":
    import functools, subprocess
    import mcp.client.stdio as mcp_stdio
    mcp_stdio.stdio_client = functools.partial(mcp_stdio.stdio_client, errlog=subprocess.DEVNULL)


## The accounts module, as a plain Python class

Before it's an MCP server, it's just a class — this is the "someone already
wrote the business logic" step from the original course.


In [ ]:
from backend.accounts import Account

warren = Account.get("Warren")
print(warren.report())
print(warren.buy_shares("AAPL", 5, "Starting position in a stable blue-chip"))
print(warren.report())


## The same class, wrapped as an MCP server

`FastMCP` turns each decorated function into a tool an agent can discover
and call — `backend/accounts_server.py` does exactly this for every method
on `Account`.


In [ ]:
with open("backend/accounts_server.py") as f:
    print(f.read()[:900])


## Launching your server and listing its tools

In [ ]:
account_params = {"command": "python", "args": ["-m", "backend.accounts_server"]}

async with MCPServerStdio(params=account_params, client_session_timeout_seconds=30) as server:
    tools = await server.list_tools()

for t in tools:
    print(t.name, "-", t.description)


## Handing it to an agent

In [ ]:
async with MCPServerStdio(params=account_params, client_session_timeout_seconds=30) as server:
    agent = Agent(
        name="account_manager",
        instructions=(
            "You manage a trading account named Warren. Use your tools to check the balance, "
            "buy or sell shares, and report on the account. Always give a rationale when trading."
        ),
        model=MODEL_NAME,
        mcp_servers=[server],
    )

    result = await Runner.run(agent, "Check Warren's balance, then buy 3 shares of GOOG with a short rationale.")
    print(result.final_output)


## Resources: read-only context, not actions

Alongside tools, MCP supports **resources** — read-only data an agent (or a
human) can fetch without it looking like a callable action. We exposed one:
`accounts://{name}/report`.


In [ ]:
async with MCPServerStdio(params=account_params, client_session_timeout_seconds=30) as server:
    resources = await server.list_resources()
    for r in resources:
        print(r.uri, "-", r.name)


## Recap, and where we are heading

You wrapped a plain Python class as an MCP server with `FastMCP`, listed its
tools and resources, and handed it to a local-model agent that used it to
manage a trading account.

Tomorrow: context engineering — giving agents web search and long-term
memory so they can actually research before they trade.

## Exercise
Add a new method to `Account` (e.g. `get_profit_loss()`) and expose it as a
tool in `accounts_server.py`. Confirm your new tool shows up when you list
tools again, then have the agent use it.
